# Bluestock Mutual Fund Capstone — Day 7: Submission Self-Review

This notebook programmatically checks that all capstone deliverables exist, processed CSV reports are complete, chart files are generated, and database table row counts match the verified data anchors exactly before submission.

In [1]:
import os
import sqlite3
from pathlib import Path

ROOT = Path('..')
PROCESSED = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reports'
CHARTS = REPORTS / 'charts'
DB_PATH = ROOT / 'data' / 'db' / 'bluestock_mf.db'

print('Workspace paths initialized.')

Workspace paths initialized.


In [2]:
deliverables = [
    REPORTS / 'Final_Report.pdf',
    REPORTS / 'Bluestock_MF_Presentation.pptx',
    REPORTS / 'fund_scorecard_formatted.xlsx',
    ROOT / 'dashboard' / 'app.py',
    ROOT / 'scripts' / 'recommender.py',
    ROOT / 'scripts' / 'run_pipeline.py',
    ROOT / 'notebooks' / '03_eda_analysis.ipynb',
    ROOT / 'notebooks' / '04_performance_analytics.ipynb',
    ROOT / 'notebooks' / '05_advanced_analytics.ipynb'
]

missing_deliv = []
print('--- CHECKING DELIVERABLES ---')
for f in deliverables:
    rel = f.relative_to(ROOT)
    if f.exists():
        print(f'[OK] {rel}')
    else:
        print(f'[MISSING] {rel}')
        missing_deliv.append(rel)
print('All deliverables check complete.')

--- CHECKING DELIVERABLES ---
[OK] reports/Final_Report.pdf
[OK] reports/Bluestock_MF_Presentation.pptx
[OK] reports/fund_scorecard_formatted.xlsx
[OK] dashboard/app.py
[OK] scripts/recommender.py
[OK] scripts/run_pipeline.py
[OK] notebooks/03_eda_analysis.ipynb
[OK] notebooks/04_performance_analytics.ipynb
[OK] notebooks/05_advanced_analytics.ipynb
All deliverables check complete.


In [3]:
csv_files = [
    'clean_fund_master.csv',
    'clean_nav.csv',
    'clean_aum_by_fund_house.csv',
    'clean_sip_inflows.csv',
    'clean_category_inflows.csv',
    'clean_folio_count.csv',
    'clean_performance.csv',
    'clean_transactions.csv',
    'clean_portfolio_holdings.csv',
    'clean_benchmark_indices.csv',
    'var_cvar_report.csv',
    'cohort_analysis.csv',
    'sip_continuity.csv',
    'sector_hhi.csv',
    'fund_scorecard.csv'
]

missing_csv = []
print('--- CHECKING PROCESSED CSVs ---')
for filename in csv_files:
    path = PROCESSED / filename
    if path.exists():
        print(f'[OK] {filename} ({path.stat().st_size:,} bytes)')
    else:
        print(f'[MISSING] {filename}')
        missing_csv.append(filename)
print('Processed CSVs check complete.')

--- CHECKING PROCESSED CSVs ---
[OK] clean_fund_master.csv (6,822 bytes)
[OK] clean_nav.csv (1,219,569 bytes)
[OK] clean_aum_by_fund_house.csv (4,013 bytes)
[OK] clean_sip_inflows.csv (1,727 bytes)
[OK] clean_category_inflows.csv (3,669 bytes)
[OK] clean_folio_count.csv (821 bytes)
[OK] clean_performance.csv (6,592 bytes)
[OK] clean_transactions.csv (3,127,924 bytes)
[OK] clean_portfolio_holdings.csv (23,909 bytes)
[OK] clean_benchmark_indices.csv (250,977 bytes)
[OK] var_cvar_report.csv (4,191 bytes)
[OK] cohort_analysis.csv (312 bytes)
[OK] sip_continuity.csv (56,925 bytes)
[OK] sector_hhi.csv (2,818 bytes)
[OK] fund_scorecard.csv (6,188 bytes)
Processed CSVs check complete.


In [4]:
expected_tables = {
    'dim_fund': 40,
    'fact_transactions': 32778,
    'fact_performance': 40,
    'fact_portfolio': 322,
    'fact_aum': 90,
    'fact_sip_industry': 48,
    'fact_nav': 46000 # Minimum limit for check
}

missing_db = []
print('--- CHECKING SQLITE DATABASE ---')
if not DB_PATH.exists():
    print(f'[MISSING] Database file at {DB_PATH.relative_to(ROOT)}')
    missing_db.append('db_file')
else:
    conn = sqlite3.connect(str(DB_PATH))
    cursor = conn.cursor()
    for table, count in expected_tables.items():
        try:
            cursor.execute(f'SELECT COUNT(*) FROM {table}')
            real_count = cursor.fetchone()[0]
            if table == 'fact_nav':
                if real_count >= count:
                    print(f'[OK] Table {table}: {real_count} rows (Expected >= {count})')
                else:
                    print(f'[FAIL] Table {table}: {real_count} rows (Expected >= {count})')
                    missing_db.append(table)
            else:
                if real_count == count:
                    print(f'[OK] Table {table}: {real_count} rows (Expected: {count})')
                else:
                    print(f'[FAIL] Table {table}: {real_count} rows (Expected: {count})')
                    missing_db.append(table)
        except Exception as e:
            print(f'[FAIL] Table {table}: Error: {e}')
            missing_db.append(table)
    conn.close()
print('Database check complete.')

--- CHECKING SQLITE DATABASE ---
[OK] Table dim_fund: 40 rows (Expected: 40)
[OK] Table fact_transactions: 32778 rows (Expected: 32778)
[OK] Table fact_performance: 40 rows (Expected: 40)
[OK] Table fact_portfolio: 322 rows (Expected: 322)
[OK] Table fact_aum: 90 rows (Expected: 90)
[OK] Table fact_sip_industry: 48 rows (Expected: 48)
[OK] Table fact_nav: 46000 rows (Expected >= 46000)
Database check complete.


In [5]:
expected_charts = [
    # EDA Charts (Day 3)
    *[f'chart_{i:02d}_nav_trends_all_funds.png' if i==1 else f'chart_{i:02d}_aum_growth_by_amc.png' if i==2 else f'chart_{i:02d}_sip_inflow_trend.png' if i==3 else f'chart_{i:02d}_category_inflow_heatmap.png' if i==4 else f'chart_{i:02d}_investor_demographics.png' if i==5 else f'chart_{i:02d}_geographic_distribution.png' if i==6 else f'chart_{i:02d}_folio_count_growth.png' if i==7 else f'chart_{i:02d}_risk_return_matrix.png' if i==8 else f'chart_{i:02d}_nav_correlation_matrix.png' if i==9 else f'chart_{i:02d}_sector_allocation.png' if i==10 else f'chart_{i:02d}_sip_accounts_dual.png' if i==11 else f'chart_{i:02d}_fund_house_market_share.png' if i==12 else f'chart_{i:02d}_category_inflow_fy25.png' if i==13 else f'chart_{i:02d}_monthly_tx_volume.png' if i==14 else f'chart_{i:02d}_benchmark_performance_indexed.png' for i in range(1, 16)],
    # Day 4 & Day 6 Charts
    'chart_16_benchmark_comparison.png',
    'chart_17_rolling_sharpe.png',
    'chart_18_scorecard_heatmap.png',
    'chart_19_var_distribution.png',
    'chart_20_rolling_sharpe.png',
    'chart_21_cohort_analysis.png',
    'chart_22_sector_hhi.png',
    'chart_23_sip_heat_calendar.png',
    # Dashboard static panels
    'dashboard_page1_industry_overview.png',
    'dashboard_page2_fund_performance.png',
    'dashboard_page3_investor_analytics.png',
    'dashboard_page4_sip_trends.png'
]

missing_charts = []
print('--- CHECKING VISUAL ASSETS ---')
for chart_name in expected_charts:
    path = CHARTS / chart_name
    if path.exists():
        print(f'[OK] {chart_name}')
    else:
        print(f'[MISSING] {chart_name}')
        missing_charts.append(chart_name)
print('Visual assets check complete.')

--- CHECKING VISUAL ASSETS ---
[OK] chart_01_nav_trends_all_funds.png
[OK] chart_02_aum_growth_by_amc.png
[OK] chart_03_sip_inflow_trend.png
[OK] chart_04_category_inflow_heatmap.png
[OK] chart_05_investor_demographics.png
[OK] chart_06_geographic_distribution.png
[OK] chart_07_folio_count_growth.png
[OK] chart_08_risk_return_matrix.png
[OK] chart_09_nav_correlation_matrix.png
[OK] chart_10_sector_allocation.png
[OK] chart_11_sip_accounts_dual.png
[OK] chart_12_fund_house_market_share.png
[OK] chart_13_category_inflow_fy25.png
[OK] chart_14_monthly_tx_volume.png
[OK] chart_15_benchmark_performance_indexed.png
[OK] chart_16_benchmark_comparison.png
[OK] chart_17_rolling_sharpe.png
[OK] chart_18_scorecard_heatmap.png
[OK] chart_19_var_distribution.png
[OK] chart_20_rolling_sharpe.png
[OK] chart_21_cohort_analysis.png
[OK] chart_22_sector_hhi.png
[OK] chart_23_sip_heat_calendar.png
[OK] dashboard_page1_industry_overview.png
[OK] dashboard_page2_fund_performance.png
[OK] dashboard_page3_in

In [6]:
is_ready = len(missing_deliv) == 0 and len(missing_csv) == 0 and len(missing_db) == 0 and len(missing_charts) == 0
if is_ready:
    print('============================================================')
    print('STATUS: READY FOR SUBMISSION')
    print('All checks passed successfully.')
    print('============================================================')
else:
    print('============================================================')
    print('STATUS: NOT READY')
    print(f'Missing deliverables: {len(missing_deliv)}')
    print(f'Missing processed CSVs: {len(missing_csv)}')
    print(f'Missing database tables: {len(missing_db)}')
    print(f'Missing charts: {len(missing_charts)}')
    print('============================================================')

STATUS: READY FOR SUBMISSION
All checks passed successfully.
